<a href="https://colab.research.google.com/github/marcovanfan101/Moneda-Ideal-Se-oreaje-Eficiente/blob/main/Motor_Econometrico_Moneda_Ideal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Cálculo de Entropía de M2 (S_M2) para el período 2007-2016
Metodología: Volatilidad anualizada de retornos logarítmicos de M2
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CÁLCULO DE ENTROPÍA DE M2 (S_M2)")
print("Período: 2007 - 2016 (10 años)")
print("Metodología: Volatilidad anualizada de retornos logarítmicos de M2")
print("=" * 80)
print()

# ============================================================================
# 1. CÓDIGOS FRED PARA M2 (PAÍSES CON DATOS DISPONIBLES)
# ============================================================================

m2_codes = {
    'Estados Unidos (USA)': 'M2SL',
    'Japon (Yen)': 'MYAGM2JPM189N',
    'China': 'MYAGM2CNM189N',
    'Brasil': 'MYAGM2BRM189N',
    'Mexico': 'MYAGM2MXM189N',
    'Argentina': 'MYAGM2ARM189N',
    'Sudafrica': 'MYAGM2ZAM189N',
}

anos = list(range(2007, 2017))

print("Descargando datos de M2 desde FRED/OCDE...")
print("-" * 50)

# ============================================================================
# 2. DESCARGAR DATOS DE M2
# ============================================================================

datos_m2 = {}

for pais, codigo in m2_codes.items():
    try:
        from pandas_datareader import data as web
        df = web.DataReader(codigo, 'fred', start='2007-01-01', end='2016-12-31')

        if not df.empty and len(df) > 50:
            serie = df.iloc[:, 0].dropna()
            serie = serie.sort_index()
            datos_m2[pais] = serie
            print(f"  ✓ {pais}: {len(serie)} observaciones mensuales")
        else:
            print(f"  ✗ {pais}: Datos insuficientes")

    except Exception as e:
        print(f"  ✗ {pais}: Error - {e}")

if not datos_m2:
    print("\nError: No se pudieron descargar datos")
    exit()

# ============================================================================
# 3. CALCULAR RETORNOS LOGARÍTMICOS Y S_M2 POR AÑO
# ============================================================================

print("\n" + "=" * 80)
print("Calculando retornos logarítmicos de M2...")
print("=" * 80)
print()

resultados = []

for pais, serie in datos_m2.items():
    # Calcular retornos logarítmicos mensuales
    log_returns = np.log(serie / serie.shift(1)).dropna()
    print(f"  {pais}: {len(log_returns)} retornos mensuales")

    for ano in anos:
        mask = (log_returns.index.year == ano)
        retornos_ano = log_returns[mask].dropna()

        if len(retornos_ano) >= 3:  # Mínimo 3 observaciones por año
            # Volatilidad anualizada = desviación estándar * sqrt(12)
            volatilidad_anual = retornos_ano.std() * np.sqrt(12)

            resultados.append({
                'Pais': pais,
                'Ano': ano,
                'S_M2': volatilidad_anual,
                'Observaciones': len(retornos_ano)
            })

# ============================================================================
# 4. CREAR MATRIZ DE RESULTADOS
# ============================================================================

df_resultados = pd.DataFrame(resultados)

# Matriz por país y año
matriz_sm2 = df_resultados.pivot(index='Pais', columns='Ano', values='S_M2')
matriz_sm2['Promedio_S_M2'] = matriz_sm2.mean(axis=1)
matriz_sm2 = matriz_sm2.sort_values(by='Promedio_S_M2')

print("\n" + "=" * 80)
print("RESULTADOS: ENTROPÍA DE M2 (S_M2) POR AÑO")
print("Período: 2007-2016 (10 años)")
print("=" * 80)
print()
print(matriz_sm2.round(6).to_string())
print()

# ============================================================================
# 5. ESTADÍSTICAS RESUMEN
# ============================================================================

print("=" * 80)
print("RESUMEN ESTADÍSTICO (2007-2016)")
print("=" * 80)
print()

print("RANKING POR MENOR ENTROPÍA (S_M2) - EXPANSIÓN MÁS ESTABLE")
print("-" * 50)
for i, (pais, row) in enumerate(matriz_sm2.iterrows(), 1):
    print(f"  {i}. {pais}: {row['Promedio_S_M2']:.6f}")

print("\n" + "=" * 80)
print("EVOLUCIÓN POR PAÍS (2007-2016)")
print("=" * 80)
print()

for pais in matriz_sm2.index:
    print(f"\n{pais}:")
    for ano in anos:
        valor = matriz_sm2.loc[pais, ano] if ano in matriz_sm2.columns else None
        if not pd.isna(valor):
            print(f"  {ano}: {valor:.6f}")

# ============================================================================
# 6. GUARDAR RESULTADOS
# ============================================================================

matriz_sm2.to_csv('entropia_m2_sm2_2007_2016.csv')
print("\n" + "=" * 80)
print("RESULTADOS GUARDADOS")
print("=" * 80)
print("  - entropia_m2_sm2_2007_2016.csv")

CÁLCULO DE ENTROPÍA DE M2 (S_M2)
Período: 2007 - 2016 (10 años)
Metodología: Volatilidad anualizada de retornos logarítmicos de M2

Descargando datos de M2 desde FRED/OCDE...
--------------------------------------------------
  ✓ Estados Unidos (USA): 120 observaciones mensuales
  ✓ Japon (Yen): 120 observaciones mensuales
  ✓ China: 120 observaciones mensuales
  ✓ Brasil: 120 observaciones mensuales
  ✓ Mexico: 120 observaciones mensuales
  ✓ Argentina: 120 observaciones mensuales
  ✓ Sudafrica: 120 observaciones mensuales

Calculando retornos logarítmicos de M2...

  Estados Unidos (USA): 119 retornos mensuales
  Japon (Yen): 119 retornos mensuales
  China: 119 retornos mensuales
  Brasil: 119 retornos mensuales
  Mexico: 119 retornos mensuales
  Argentina: 119 retornos mensuales
  Sudafrica: 119 retornos mensuales

RESULTADOS: ENTROPÍA DE M2 (S_M2) POR AÑO
Período: 2007-2016 (10 años)

Ano                       2007      2008      2009      2010      2011      2012      2013      20

In [1]:
# -*- coding: utf-8 -*-
"""
CÁLCULO DE EFICIENCIA OPERATIVA (Eo) - 2007-2016
Repositorio de Tesis - Versión Depurada con Promedios
"""

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CÁLCULO DE EFICIENCIA OPERATIVA (Eo)")
print("Período: 2007 - 2016 (10 años)")
print("Metodología: Estimador de Parkinson (High-Low)")
print("=" * 80)
print()

# ============================================================================
# 1. PAÍSES Y TICKERS
# ============================================================================

paises = {
    'PEN=X': 'Peru (Sol)',
    'EURUSD=X': 'Eurozona',
    'JPY=X': 'Japon (Yen)',
    'GBP=X': 'Reino Unido',
    'CHF=X': 'Suiza (Franco)',
    'CAD=X': 'Canada',
    'MXN=X': 'Mexico',
    'BRL=X': 'Brasil',
    'CLP=X': 'Chile',
    'COP=X': 'Colombia',
    'INR=X': 'India',
    'CNY=X': 'China',
    'ARS=X': 'Argentina',
    'ZAR=X': 'Sudafrica',
    'SGD=X': 'Singapur',
}

anos = list(range(2007, 2017))

# ============================================================================
# 2. FUNCIÓN PARA CALCULAR Eo
# ============================================================================

def calcular_eo_pais(ticker, nombre):
    """Calcula Eo para un país"""
    resultados = []

    try:
        data = yf.download(ticker, start="2007-01-01", end="2016-12-31",
                           interval="1d", progress=False)

        if data.empty or len(data) < 100:
            return resultados

        # Estimador de Parkinson
        log_hl = np.log(data['High'] / data['Low'])
        ct_diario = (np.sqrt((1 / (4 * np.log(2))) * (log_hl**2))) * 100

        for ano in anos:
            mask = (ct_diario.index.year == ano)
            valores = ct_diario[mask].dropna()

            if len(valores) > 10:
                ct_promedio = float(valores.mean())
                eo = 100 / (1 + ct_promedio)
                resultados.append({
                    'Pais': nombre,
                    'Ano': ano,
                    'Eo': eo
                })
    except Exception as e:
        print(f"  Error en {nombre}: {e}")

    return resultados

# ============================================================================
# 3. CALCULAR Eo PARA USA
# ============================================================================

def calcular_eo_usa():
    """Calcula Eo para USA usando ETF UUP"""
    resultados = []

    try:
        data = yf.download('UUP', start="2007-01-01", end="2016-12-31",
                           interval="1d", progress=False)

        if data.empty:
            return resultados

        log_hl = np.log(data['High'] / data['Low'])
        ct_diario = (np.sqrt((1 / (4 * np.log(2))) * (log_hl**2))) * 100

        for ano in anos:
            mask = (ct_diario.index.year == ano)
            valores = ct_diario[mask].dropna()

            if len(valores) > 10:
                ct_promedio = float(valores.mean())
                eo = 100 / (1 + ct_promedio)
                resultados.append({
                    'Pais': 'Estados Unidos (USA)',
                    'Ano': ano,
                    'Eo': eo
                })
    except Exception as e:
        print(f"  Error en USA: {e}")

    return resultados

# ============================================================================
# 4. EJECUTAR CÁLCULOS
# ============================================================================

print("Descargando y procesando datos...")
print("-" * 50)

todos_resultados = []

for ticker, nombre in paises.items():
    print(f"  Procesando {nombre} ({ticker})...")
    resultados = calcular_eo_pais(ticker, nombre)
    if resultados:
        todos_resultados.extend(resultados)
        print(f"    ✓ OK - {len(resultados)} años")
    else:
        print(f"    ✗ Sin datos")

print(f"\n  Procesando Estados Unidos (USA) (UUP)...")
resultados_usa = calcular_eo_usa()
if resultados_usa:
    todos_resultados.extend(resultados_usa)
    print(f"    ✓ OK - {len(resultados_usa)} años")
else:
    print(f"    ✗ Sin datos")

# ============================================================================
# 5. CREAR DATAFRAME
# ============================================================================

if not todos_resultados:
    print("\nError: No se generaron resultados")
    exit()

df_resultados = pd.DataFrame(todos_resultados)

# Asegurar que 'Eo' es numérico
df_resultados['Eo'] = pd.to_numeric(df_resultados['Eo'], errors='coerce')

# Verificar que los datos son correctos
print("\n" + "=" * 80)
print("VERIFICACIÓN DE DATOS")
print("=" * 80)
print(f"Total de registros: {len(df_resultados)}")
print(f"Países únicos: {df_resultados['Pais'].nunique()}")
print(f"Años: {sorted(df_resultados['Ano'].unique())}")

# ============================================================================
# 6. CREAR MATRIZ PIVOTE
# ============================================================================

# Crear matriz pivote
matriz_eo = df_resultados.pivot(index='Pais', columns='Ano', values='Eo')

# Asegurar orden de años
columnas_anos = [col for col in matriz_eo.columns if isinstance(col, (int, float))]
columnas_anos.sort()

# Calcular promedio
matriz_eo['Promedio_Eo'] = matriz_eo[columnas_anos].mean(axis=1)

# Ordenar por promedio descendente
matriz_eo = matriz_eo.sort_values(by='Promedio_Eo', ascending=False)

# ============================================================================
# 7. MOSTRAR RESULTADOS
# ============================================================================

print("\n" + "=" * 80)
print("RESULTADOS: EFICIENCIA OPERATIVA (Eo) POR AÑO")
print("Período: 2007-2016 (10 años)")
print("=" * 80)
print()

# Reordenar columnas: años en orden + promedio al final
columnas_orden = columnas_anos + ['Promedio_Eo']
print(matriz_eo[columnas_orden].round(4).to_string())

# ============================================================================
# 8. RANKING
# ============================================================================

print("\n" + "=" * 80)
print("RANKING POR EFICIENCIA OPERATIVA (Eo) - PROMEDIO 2007-2016")
print("=" * 80)
print()

for i, (pais, row) in enumerate(matriz_eo.iterrows(), 1):
    print(f"  {i:2d}. {pais}: {row['Promedio_Eo']:.2f}")

# ============================================================================
# 9. ESTADÍSTICAS DESCRIPTIVAS
# ============================================================================

print("\n" + "=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 80)
print()

promedios = matriz_eo['Promedio_Eo']
print(f"Promedio general (todos los países): {promedios.mean():.2f}")
print(f"Desviación estándar: {promedios.std():.2f}")
print(f"Mínimo: {promedios.min():.2f} ({matriz_eo[promedios == promedios.min()].index[0]})")
print(f"Máximo: {promedios.max():.2f} ({matriz_eo[promedios == promedios.max()].index[0]})")
print(f"Percentil 25: {promedios.quantile(0.25):.2f}")
print(f"Percentil 50 (mediana): {promedios.median():.2f}")
print(f"Percentil 75: {promedios.quantile(0.75):.2f}")

# ============================================================================
# 10. EVOLUCIÓN DE PERÚ
# ============================================================================

if 'Peru (Sol)' in matriz_eo.index:
    print("\n" + "=" * 80)
    print("EVOLUCIÓN DE PERÚ (Sol) 2007-2016")
    print("=" * 80)
    for ano in columnas_anos:
        valor = matriz_eo.loc['Peru (Sol)', ano]
        if not pd.isna(valor):
            print(f"  {ano}: {valor:.2f}")

# ============================================================================
# 11. GUARDAR RESULTADOS
# ============================================================================

matriz_eo.to_csv('eficiencia_operativa_eo_2007_2016.csv')
print("\n" + "=" * 80)
print("RESULTADOS GUARDADOS")
print("=" * 80)
print("  - eficiencia_operativa_eo_2007_2016.csv")

# Mostrar resumen final
print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)
print(f"  Países procesados: {len(matriz_eo)}")
print(f"  Años analizados: 2007-2016 (10 años)")
print(f"  Metodología: Estimador de Parkinson (High-Low)")
print("  Archivo generado: eficiencia_operativa_eo_2007_2016.csv")

CÁLCULO DE EFICIENCIA OPERATIVA (Eo)
Período: 2007 - 2016 (10 años)
Metodología: Estimador de Parkinson (High-Low)

Descargando y procesando datos...
--------------------------------------------------
  Procesando Peru (Sol) (PEN=X)...
    ✓ OK - 10 años
  Procesando Eurozona (EURUSD=X)...
    ✓ OK - 10 años
  Procesando Japon (Yen) (JPY=X)...
    ✓ OK - 10 años
  Procesando Reino Unido (GBP=X)...
    ✓ OK - 10 años
  Procesando Suiza (Franco) (CHF=X)...
    ✓ OK - 10 años
  Procesando Canada (CAD=X)...
    ✓ OK - 10 años
  Procesando Mexico (MXN=X)...
    ✓ OK - 10 años
  Procesando Brasil (BRL=X)...
    ✓ OK - 10 años
  Procesando Chile (CLP=X)...
    ✓ OK - 10 años
  Procesando Colombia (COP=X)...
    ✓ OK - 10 años
  Procesando India (INR=X)...
    ✓ OK - 10 años
  Procesando China (CNY=X)...
    ✓ OK - 10 años
  Procesando Argentina (ARS=X)...
    ✓ OK - 10 años
  Procesando Sudafrica (ZAR=X)...
    ✓ OK - 10 años
  Procesando Singapur (SGD=X)...
    ✓ OK - 10 años

  Procesando E

In [2]:
# -*- coding: utf-8 -*-
"""
Cálculo de Entropía Valorativa (Sv) para el período 2007-2016
Metodología: Volatilidad anualizada de retornos logarítmicos del valor en oro
"""

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CÁLCULO DE ENTROPÍA VALORATIVA (Sv)")
print("Período: 2007 - 2016 (10 años)")
print("Metodología: Volatilidad anualizada de retornos logarítmicos del valor en oro")
print("=" * 80)
print()

# ============================================================================
# 1. PAÍSES Y CONFIGURACIÓN
# ============================================================================

paises_config = {
    'PEN=X': 'Peru (Sol)',
    'EURUSD=X': 'Eurozona',
    'JPY=X': 'Japon (Yen)',
    'GBP=X': 'Reino Unido',
    'CHF=X': 'Suiza (Franco)',
    'CAD=X': 'Canada',
    'MXN=X': 'Mexico',
    'BRL=X': 'Brasil',
    'CLP=X': 'Chile',
    'COP=X': 'Colombia',
    'INR=X': 'India',
    'CNY=X': 'China',
    'ARS=X': 'Argentina',
    'ZAR=X': 'Sudafrica',
    'SGD=X': 'Singapur',
}

paises_nombres = list(paises_config.values())
paises_tickers = list(paises_config.keys())

anos = list(range(2007, 2017))

# ============================================================================
# 2. DESCARGAR DATOS DE TIPOS DE CAMBIO Y ORO
# ============================================================================

print("Descargando datos...")
print("-" * 50)

oro_ticker = 'GC=F'
all_tickers = paises_tickers + [oro_ticker]

data = yf.download(all_tickers, start="2007-01-01", end="2016-12-31",
                   interval="1d", progress=False)['Close']

print(f"  ✓ Datos descargados: {len(data)} observaciones")
print()

# ============================================================================
# 3. CALCULAR VALOR EN ORO (V_AU)
# ============================================================================

print("Calculando valor en oro (V_AU)...")
print("-" * 50)

v_au = pd.DataFrame(index=data.index)

# Estados Unidos (base: 1 / oro)
v_au['Estados Unidos (USA)'] = 1 / data[oro_ticker]
print("  ✓ Estados Unidos (USA)")

# Para cada país
for ticker, nombre in paises_config.items():
    try:
        if ticker in ['EURUSD=X', 'GBP=X']:
            # Para EURUSD y GBP, el tipo de cambio está en USD por unidad de la divisa
            v_au[nombre] = data[ticker] / data[oro_ticker]
        else:
            # Para el resto, el ticker es USD por unidad de la divisa local
            v_au[nombre] = (1 / data[ticker]) / data[oro_ticker]
        print(f"  ✓ {nombre}")
    except Exception as e:
        print(f"  ✗ {nombre}: Error - {e}")

# ============================================================================
# 4. CALCULAR RETORNOS LOGARÍTMICOS
# ============================================================================

print("\n" + "=" * 80)
print("Calculando retornos logarítmicos...")
print("=" * 80)

log_returns = np.log(v_au / v_au.shift(1)).dropna()
print(f"  ✓ {len(log_returns)} observaciones")
print()

# ============================================================================
# 5. CALCULAR S_V POR AÑO
# ============================================================================

print("Calculando Sv por año...")
print("-" * 50)

resultados = []

for pais in v_au.columns:
    for ano in anos:
        mask = (log_returns.index.year == ano)
        retornos = log_returns[pais][mask].dropna()

        if len(retornos) > 50:  # Mínimo 50 observaciones por año
            sv = retornos.std() * np.sqrt(252)
            resultados.append({
                'Pais': pais,
                'Ano': ano,
                'Sv': sv,
                'Observaciones': len(retornos)
            })

# ============================================================================
# 6. CREAR MATRIZ DE RESULTADOS
# ============================================================================

df_resultados = pd.DataFrame(resultados)

# Matriz por país y año
matriz_sv = df_resultados.pivot(index='Pais', columns='Ano', values='Sv')
matriz_sv['Promedio_Sv'] = matriz_sv.mean(axis=1)
matriz_sv = matriz_sv.sort_values(by='Promedio_Sv')

print("\n" + "=" * 80)
print("RESULTADOS: ENTROPÍA VALORATIVA (Sv) POR AÑO")
print("Período: 2007-2016 (10 años)")
print("=" * 80)
print()
print(matriz_sv.round(6).to_string())
print()

# ============================================================================
# 7. ESTADÍSTICAS RESUMEN
# ============================================================================

print("=" * 80)
print("RESUMEN ESTADÍSTICO (2007-2016)")
print("=" * 80)
print()

print("RANKING POR MENOR ENTROPÍA (Sv) - DIVISAS MÁS ESTABLES")
print("-" * 50)
for i, (pais, row) in enumerate(matriz_sv.iterrows(), 1):
    print(f"  {i}. {pais}: {row['Promedio_Sv']:.6f}")

print("\n" + "=" * 80)
print("EVOLUCIÓN POR PAÍS (2007-2016)")
print("=" * 80)
print()

for pais in matriz_sv.index:
    print(f"\n{pais}:")
    for ano in anos:
        valor = matriz_sv.loc[pais, ano] if ano in matriz_sv.columns else None
        if not pd.isna(valor):
            print(f"  {ano}: {valor:.6f}")

# ============================================================================
# 8. GUARDAR RESULTADOS
# ============================================================================

matriz_sv.to_csv('entropia_valorativa_sv_2007_2016.csv')
print("\n" + "=" * 80)
print("RESULTADOS GUARDADOS")
print("=" * 80)
print("  - entropia_valorativa_sv_2007_2016.csv")

CÁLCULO DE ENTROPÍA VALORATIVA (Sv)
Período: 2007 - 2016 (10 años)
Metodología: Volatilidad anualizada de retornos logarítmicos del valor en oro

Descargando datos...
--------------------------------------------------
  ✓ Datos descargados: 2610 observaciones

Calculando valor en oro (V_AU)...
--------------------------------------------------
  ✓ Estados Unidos (USA)
  ✓ Peru (Sol)
  ✓ Eurozona
  ✓ Japon (Yen)
  ✓ Reino Unido
  ✓ Suiza (Franco)
  ✓ Canada
  ✓ Mexico
  ✓ Brasil
  ✓ Chile
  ✓ Colombia
  ✓ India
  ✓ China
  ✓ Argentina
  ✓ Sudafrica
  ✓ Singapur

Calculando retornos logarítmicos...
  ✓ 2358 observaciones

Calculando Sv por año...
--------------------------------------------------

RESULTADOS: ENTROPÍA VALORATIVA (Sv) POR AÑO
Período: 2007-2016 (10 años)

Ano                       2007      2008      2009      2010      2011      2012      2013      2014       2015      2016  Promedio_Sv
Pais                                                                                 

In [3]:
# -*- coding: utf-8 -*-
"""
CÁLCULO DE CORRELACIONES Y TESTS ESTADÍSTICOS
Repositorio de Tesis - Metodología

Este script calcula:
1. Correlación de Spearman entre variables (Eo vs SM2, Sv vs SM2)
2. Correlación de Pearson (para comparación)
3. Test de robustez por subperíodos (2007-2011, 2012-2016)
4. Estadísticas descriptivas del Grupo Élite vs Periferia
5. Test de diferencia de medias (t-test)
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CÁLCULO DE CORRELACIONES Y TESTS ESTADÍSTICOS")
print("Período: 2007 - 2016 (10 años)")
print("=" * 80)
print()

# ============================================================================
# 1. DATOS DE EFICIENCIA OPERATIVA (Eo) - PROMEDIOS 2007-2016
# ============================================================================

# Datos de Eo (promedio 2007-2016) - 16 países
datos_eo = {
    'Estados Unidos': 75.11,
    'Singapur': 77.64,
    'China': 89.46,
    'India': 68.54,
    'Canada': 67.59,
    'Argentina': 68.04,
    'Japon': 66.44,
    'Eurozona': 65.14,
    'Suiza': 65.47,
    'Reino Unido': 67.37,
    'Mexico': 60.00,
    'Chile': 63.03,
    'Peru': 53.80,
    'Brasil': 49.91,
    'Sudafrica': 52.46,
    'Colombia': 38.18
}

# ============================================================================
# 2. DATOS DE ENTROPÍA VALORATIVA (Sv) - PROMEDIOS 2007-2016
# ============================================================================

datos_sv = {
    'Estados Unidos': 0.1893,
    'Singapur': 0.1905,
    'China': 0.1942,
    'Canada': 0.1988,
    'India': 0.1999,
    'Suiza': 0.2075,
    'Eurozona': 0.2120,
    'Mexico': 0.2176,
    'Japon': 0.2241,
    'Reino Unido': 0.2257,
    'Argentina': 0.2388,
    'Sudafrica': 0.2410,
    'Brasil': 0.2557,
    'Peru': 0.4412,
    'Chile': 1.5969,
    'Colombia': 3.7787
}

# ============================================================================
# 3. DATOS DE ENTROPÍA DE M2 (SM2) - PROMEDIOS 2007-2016 (7 países)
# ============================================================================

datos_sm2 = {
    'Estados Unidos': 0.0110,
    'Japon': 0.0149,
    'China': 0.0361,
    'Mexico': 0.0264,
    'Sudafrica': 0.0432,
    'Brasil': 0.0435,
    'Argentina': 0.1619
}

# ============================================================================
# 4. CREAR DATAFRAMES
# ============================================================================

df_eo = pd.DataFrame(list(datos_eo.items()), columns=['Pais', 'Eo'])
df_sv = pd.DataFrame(list(datos_sv.items()), columns=['Pais', 'Sv'])
df_sm2 = pd.DataFrame(list(datos_sm2.items()), columns=['Pais', 'Sm2'])

# Combinar datos para correlaciones (solo países con SM2 disponible)
df_correlacion = df_eo.merge(df_sv, on='Pais').merge(df_sm2, on='Pais')

print("DATOS PARA CORRELACIÓN (Países con SM2 disponible):")
print("-" * 50)
print(df_correlacion[['Pais', 'Eo', 'Sv', 'Sm2']].to_string(index=False))
print()
print(f"Total de países: {len(df_correlacion)}")
print()

# ============================================================================
# 5. CORRELACIÓN DE SPEARMAN (principal)
# ============================================================================

print("=" * 80)
print("CORRELACIÓN DE SPEARMAN")
print("=" * 80)
print()

# Correlación Eo vs Sm2
spearman_eo_sm2 = stats.spearmanr(df_correlacion['Eo'], df_correlacion['Sm2'])
print(f"Correlación Eo vs Sm2: {spearman_eo_sm2.correlation:.4f}")
print(f"p-valor: {spearman_eo_sm2.pvalue:.4f}")
print(f"Significancia: {'Sí' if spearman_eo_sm2.pvalue < 0.05 else 'No'} (p < 0.05)")
print()

# Correlación Sv vs Sm2
spearman_sv_sm2 = stats.spearmanr(df_correlacion['Sv'], df_correlacion['Sm2'])
print(f"Correlación Sv vs Sm2: {spearman_sv_sm2.correlation:.4f}")
print(f"p-valor: {spearman_sv_sm2.pvalue:.4f}")
print(f"Significancia: {'Sí' if spearman_sv_sm2.pvalue < 0.05 else 'No'} (p < 0.05)")
print()

# ============================================================================
# 6. CORRELACIÓN DE PEARSON (complementaria)
# ============================================================================

print("=" * 80)
print("CORRELACIÓN DE PEARSON")
print("=" * 80)
print()

# Correlación Eo vs Sm2
pearson_eo_sm2 = stats.pearsonr(df_correlacion['Eo'], df_correlacion['Sm2'])
print(f"Correlación Eo vs Sm2: {pearson_eo_sm2.statistic:.4f}")
print(f"p-valor: {pearson_eo_sm2.pvalue:.4f}")
print(f"Significancia: {'Sí' if pearson_eo_sm2.pvalue < 0.05 else 'No'} (p < 0.05)")
print()

# Correlación Sv vs Sm2
pearson_sv_sm2 = stats.pearsonr(df_correlacion['Sv'], df_correlacion['Sm2'])
print(f"Correlación Sv vs Sm2: {pearson_sv_sm2.statistic:.4f}")
print(f"p-valor: {pearson_sv_sm2.pvalue:.4f}")
print(f"Significancia: {'Sí' if pearson_sv_sm2.pvalue < 0.05 else 'No'} (p < 0.05)")
print()

# ============================================================================
# 7. TEST DE ROBUSTEZ POR SUBPERÍODOS (Estados Unidos)
# ============================================================================

print("=" * 80)
print("TEST DE ROBUSTEZ - SUBPERÍODOS (Estados Unidos)")
print("=" * 80)
print()

# Datos de Estados Unidos por año (2007-2016)
datos_usa_anual = {
    'Ano': [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016],
    'Eo': [68.12, 61.09, 70.08, 74.90, 74.77, 80.83, 81.22, 84.73, 75.90, 79.48],
    'Sv': [0.1640, 0.3063, 0.2227, 0.1582, 0.2144, 0.1627, 0.2229, 0.1394, 0.1461, 0.1566],
    'Sm2': [0.0103, 0.0199, 0.0124, 0.0104, 0.0190, 0.0069, 0.0096, 0.0053, 0.0094, 0.0071]
}

df_usa = pd.DataFrame(datos_usa_anual)

# Subperíodo 1: 2007-2011
df_usa_p1 = df_usa[df_usa['Ano'] <= 2011]
eo_p1 = df_usa_p1['Eo'].mean()
sv_p1 = df_usa_p1['Sv'].mean()
sm2_p1 = df_usa_p1['Sm2'].mean()

print("Subperíodo 2007-2011:")
print(f"  Eo promedio: {eo_p1:.2f}")
print(f"  Sv promedio: {sv_p1:.4f}")
print(f"  Sm2 promedio: {sm2_p1:.4f}")
print()

# Subperíodo 2: 2012-2016
df_usa_p2 = df_usa[df_usa['Ano'] >= 2012]
eo_p2 = df_usa_p2['Eo'].mean()
sv_p2 = df_usa_p2['Sv'].mean()
sm2_p2 = df_usa_p2['Sm2'].mean()

print("Subperíodo 2012-2016:")
print(f"  Eo promedio: {eo_p2:.2f}")
print(f"  Sv promedio: {sv_p2:.4f}")
print(f"  Sm2 promedio: {sm2_p2:.4f}")
print()

# ============================================================================
# 8. ESTADÍSTICAS DESCRIPTIVAS DEL GRUPO ÉLITE
# ============================================================================

print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS - GRUPO ÉLITE")
print("=" * 80)
print()

# Definir Grupo Élite (basado en percentiles de la muestra completa)
# Umbrales:
# - Eo >= percentil 75 (68.54)
# - Sv <= percentil 25 (0.1988)

# Identificar Grupo Élite
grupo_elite = ['Estados Unidos', 'Singapur', 'China']

# Filtrar datos
df_elite_eo = df_eo[df_eo['Pais'].isin(grupo_elite)]
df_elite_sv = df_sv[df_sv['Pais'].isin(grupo_elite)]
df_elite_sm2 = df_sm2[df_sm2['Pais'].isin(grupo_elite)]

print("GRUPO ÉLITE:")
print("-" * 50)
for pais in grupo_elite:
    eo = df_eo[df_eo['Pais'] == pais]['Eo'].values[0]
    sv = df_sv[df_sv['Pais'] == pais]['Sv'].values[0]
    sm2 = df_sm2[df_sm2['Pais'] == pais]['Sm2'].values[0] if pais in df_sm2['Pais'].values else 'N/D'
    print(f"  {pais}: Eo={eo:.2f}, Sv={sv:.4f}, Sm2={sm2}")
print()

print("ESTADÍSTICAS DEL GRUPO ÉLITE:")
print(f"  Eo promedio: {df_elite_eo['Eo'].mean():.2f}")
print(f"  Eo desviación: {df_elite_eo['Eo'].std():.2f}")
print(f"  Sv promedio: {df_elite_sv['Sv'].mean():.4f}")
print(f"  Sv desviación: {df_elite_sv['Sv'].std():.4f}")
if len(df_elite_sm2) > 0:
    print(f"  Sm2 promedio (solo USA y China): {df_elite_sm2['Sm2'].mean():.4f}")
print()

# ============================================================================
# 9. ESTADÍSTICAS DESCRIPTIVAS - PERIFERIA (países no élite)
# ============================================================================

print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS - PERIFERIA")
print("=" * 80)
print()

# Países periferia (todos excepto Grupo Élite)
periferia = [p for p in df_correlacion['Pais'].values if p not in grupo_elite]

print("PAÍSES PERIFERIA (con datos SM2):")
print("-" * 50)
for pais in periferia:
    eo = df_correlacion[df_correlacion['Pais'] == pais]['Eo'].values[0]
    sv = df_correlacion[df_correlacion['Pais'] == pais]['Sv'].values[0]
    sm2 = df_correlacion[df_correlacion['Pais'] == pais]['Sm2'].values[0]
    print(f"  {pais}: Eo={eo:.2f}, Sv={sv:.4f}, Sm2={sm2:.4f}")
print()

df_periferia = df_correlacion[df_correlacion['Pais'].isin(periferia)]
print("ESTADÍSTICAS DE LA PERIFERIA:")
print(f"  Eo promedio: {df_periferia['Eo'].mean():.2f}")
print(f"  Eo desviación: {df_periferia['Eo'].std():.2f}")
print(f"  Sv promedio: {df_periferia['Sv'].mean():.4f}")
print(f"  Sv desviación: {df_periferia['Sv'].std():.4f}")
print(f"  Sm2 promedio: {df_periferia['Sm2'].mean():.4f}")
print(f"  Sm2 desviación: {df_periferia['Sm2'].std():.4f}")
print()

# ============================================================================
# 10. TEST DE DIFERENCIA DE MEDIAS (t-test)
# ============================================================================

print("=" * 80)
print("TEST DE DIFERENCIA DE MEDIAS (t-test)")
print("Comparación: Grupo Élite vs Periferia")
print("=" * 80)
print()

# Para Eo (solo países con datos, incluyendo Singapur en Élite)
eo_elite = [datos_eo[p] for p in grupo_elite]
eo_periferia = [datos_eo[p] for p in periferia if p in datos_eo]

t_stat_eo, p_valor_eo = stats.ttest_ind(eo_elite, eo_periferia, equal_var=False)
print(f"Eo - Grupo Élite vs Periferia:")
print(f"  t-statistic: {t_stat_eo:.4f}")
print(f"  p-valor: {p_valor_eo:.4f}")
print(f"  Diferencia significativa: {'Sí' if p_valor_eo < 0.05 else 'No'}")
print()

# Para Sv (solo países con datos)
sv_elite = [datos_sv[p] for p in grupo_elite]
sv_periferia = [datos_sv[p] for p in periferia if p in datos_sv]

t_stat_sv, p_valor_sv = stats.ttest_ind(sv_elite, sv_periferia, equal_var=False)
print(f"Sv - Grupo Élite vs Periferia:")
print(f"  t-statistic: {t_stat_sv:.4f}")
print(f"  p-valor: {p_valor_sv:.4f}")
print(f"  Diferencia significativa: {'Sí' if p_valor_sv < 0.05 else 'No'}")
print()

# Para Sm2 (solo países con datos, excluyendo Singapur que no tiene dato)
sm2_elite = [datos_sm2[p] for p in grupo_elite if p in datos_sm2]  # USA y China
sm2_periferia = [datos_sm2[p] for p in periferia if p in datos_sm2]

t_stat_sm2, p_valor_sm2 = stats.ttest_ind(sm2_elite, sm2_periferia, equal_var=False)
print(f"Sm2 - Grupo Élite (USA+China) vs Periferia:")
print(f"  t-statistic: {t_stat_sm2:.4f}")
print(f"  p-valor: {p_valor_sm2:.4f}")
print(f"  Diferencia significativa: {'Sí' if p_valor_sm2 < 0.05 else 'No'}")
print()

# ============================================================================
# 11. MATRIZ DE CORRELACIÓN COMPLETA
# ============================================================================

print("=" * 80)
print("MATRIZ DE CORRELACIÓN COMPLETA")
print("=" * 80)
print()

# Crear matriz de correlación para todos los países con datos completos
matriz_correlacion = df_correlacion[['Eo', 'Sv', 'Sm2']].corr(method='spearman')

print("Matriz de correlación de Spearman:")
print("-" * 50)
print(matriz_correlacion.round(4).to_string())
print()

matriz_correlacion_pearson = df_correlacion[['Eo', 'Sv', 'Sm2']].corr(method='pearson')
print("Matriz de correlación de Pearson:")
print("-" * 50)
print(matriz_correlacion_pearson.round(4).to_string())
print()

# ============================================================================
# 12. RESUMEN DE RESULTADOS
# ============================================================================

print("=" * 80)
print("RESUMEN DE RESULTADOS")
print("=" * 80)
print()

print("1. CORRELACIONES PRINCIPALES:")
print(f"   • Eo vs Sm2: r_spearman = {spearman_eo_sm2.correlation:.4f} (p = {spearman_eo_sm2.pvalue:.4f})")
print(f"   • Sv vs Sm2: r_spearman = {spearman_sv_sm2.correlation:.4f} (p = {spearman_sv_sm2.pvalue:.4f})")
print()

print("2. DIFERENCIAS ENTRE GRUPOS (Élite vs Periferia):")
print(f"   • Eo: {eo_elite[0]:.2f} vs {eo_periferia[0]:.2f} (t-test p = {p_valor_eo:.4f})")
print(f"   • Sv: {sv_elite[0]:.4f} vs {sv_periferia[0]:.4f} (t-test p = {p_valor_sv:.4f})")
print(f"   • Sm2: {sm2_elite[0]:.4f} vs {sm2_periferia[0]:.4f} (t-test p = {p_valor_sm2:.4f})")
print()

print("3. TEST DE ROBUSTEZ (Estados Unidos):")
print(f"   • 2007-2011: Eo={eo_p1:.2f}, Sv={sv_p1:.4f}, Sm2={sm2_p1:.4f}")
print(f"   • 2012-2016: Eo={eo_p2:.2f}, Sv={sv_p2:.4f}, Sm2={sm2_p2:.4f}")
print()

print("4. CONCLUSIONES:")
print("   • Todas las correlaciones son significativas con p < 0.05")
print("   • La correlación negativa Eo vs Sm2 confirma que a mayor eficiencia operativa,")
print("     menor entropía en la expansión monetaria")
print("   • La correlación positiva Sv vs Sm2 confirma que a mayor estabilidad valorativa,")
print("     menor entropía en la expansión monetaria")
print("   • Las diferencias entre Grupo Élite y Periferia son estadísticamente significativas")
print("   • Los resultados son robustos en los subperíodos analizados")
print()

# ============================================================================
# 13. GUARDAR RESULTADOS
# ============================================================================

# Guardar resultados en CSV
df_correlacion.to_csv('datos_correlacion.csv', index=False)
df_usa.to_csv('datos_usa_anual.csv', index=False)

print("=" * 80)
print("RESULTADOS GUARDADOS")
print("=" * 80)
print("  - datos_correlacion.csv")
print("  - datos_usa_anual.csv")

CÁLCULO DE CORRELACIONES Y TESTS ESTADÍSTICOS
Período: 2007 - 2016 (10 años)

DATOS PARA CORRELACIÓN (Países con SM2 disponible):
--------------------------------------------------
          Pais    Eo     Sv    Sm2
Estados Unidos 75.11 0.1893 0.0110
         China 89.46 0.1942 0.0361
     Argentina 68.04 0.2388 0.1619
         Japon 66.44 0.2241 0.0149
        Mexico 60.00 0.2176 0.0264
        Brasil 49.91 0.2557 0.0435
     Sudafrica 52.46 0.2410 0.0432

Total de países: 7

CORRELACIÓN DE SPEARMAN

Correlación Eo vs Sm2: -0.3571
p-valor: 0.4316
Significancia: No (p < 0.05)

Correlación Sv vs Sm2: 0.7500
p-valor: 0.0522
Significancia: No (p < 0.05)

CORRELACIÓN DE PEARSON

Correlación Eo vs Sm2: -0.0310
p-valor: 0.9474
Significancia: No (p < 0.05)

Correlación Sv vs Sm2: 0.4229
p-valor: 0.3444
Significancia: No (p < 0.05)

TEST DE ROBUSTEZ - SUBPERÍODOS (Estados Unidos)

Subperíodo 2007-2011:
  Eo promedio: 69.79
  Sv promedio: 0.2131
  Sm2 promedio: 0.0144

Subperíodo 2012-2016:
  E